# 🛡️ RMIT Hackathon 2026 — Challenge 1: Multilingual Prompt-Attack Classification
## Chosen Approved Model: `protectai/deberta-v3-base-prompt-injection-v2`
- **Author:** Trung Quân (Deep Learning & Approved Hugging Face Specialist)
- **Notebook Naming Convention (for Submission):** `<university>_<campus>_<team>_C1` (e.g., `RMIT_SGS_<team>_C1`)
- **Backbone Source (Official Kaggle Whitelist):** `protectai/deberta-v3-base-prompt-injection-v2` (`https://www.kaggle.com/models/aisuko/deberta-v3-base-prompt-injection-v2/Transformers/default`)
- **Task Formulation:** 3-Class Classification (`0: BENIGN`, `1: JAILBREAK`, `2: PROMPT_INJECTION`)
- **Evaluation Metric:** Macro-Averaged F1 Score
- **Key Techniques:**
  1. Shared 5-Fold Stratified/Group Split (`folds.csv`) for leak-free team ensembling.
  2. Language-conditioned input prefixing: `"lang: {language} | {text}"`. 
  3. Head + Tail Token Truncation (`256 head + 256 tail = 512 tokens`) to preserve end-of-prompt injection payloads on long texts (up to 18,550 chars).
  4. Class-Weighted Cross-Entropy Loss (`weights = [1.0, 2.0, 6.0]`) to boost recall on the 7% `PROMPT_INJECTION` minority class.
  5. Out-Of-Fold (`oof_deberta.npy`, shape `14000x3`) & Test (`test_deberta.npy`, shape `6000x3`) probability export + Nelder-Mead class multiplier optimization.

In [ ]:
import os
import gc
import re
import math
import time
import random
import unicodedata
from pathlib import Path
from typing import Any, List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ============================================================================
# 1. OFFICIAL COMPETITION CONSTANTS & CONFIGURATION
# ============================================================================
CHOSEN_MODEL_NAME = "protectai/deberta-v3-base-prompt-injection-v2"
OUTPUT_PREFIX = "deberta"  # Exports oof_deberta.npy and test_deberta.npy

VALID_LABELS = ("BENIGN", "JAILBREAK", "PROMPT_INJECTION")
LABEL2ID = {label: idx for idx, label in enumerate(VALID_LABELS)}
ID2LABEL = {idx: label for idx, label in enumerate(VALID_LABELS)}

SEED = 42
N_SPLITS = 5
FOLD_TO_RUN = [0, 1, 2, 3, 4]  # Run all 5 folds for full OOF & Test ensemble

MAX_LEN = 512
HEAD_LEN = 256
TAIL_LEN = 256

TRAIN_BATCH_SIZE = 8
VALID_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4           # Effective batch size = 8 * 4 = 32 (saves 50% VRAM)
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0

# Class weights for [BENIGN (62%), JAILBREAK (31%), PROMPT_INJECTION (7%)]
CLASS_WEIGHTS = [1.0, 2.0, 6.0]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

seed_everything(SEED)
print(f"[Config] Chosen Approved Model : {CHOSEN_MODEL_NAME}")
print(f"[Config] Device               : {DEVICE} (CUDA count: {torch.cuda.device_count()})")
print(f"[Config] Label Mapping        : {LABEL2ID}")

In [ ]:
# ============================================================================
# 2. AUTOMATIC PATH DISCOVERY (WORKS ON KAGGLE & LOCAL REPO)
# ============================================================================
def find_file_in_dirs(filename: str, search_roots: List[Path]) -> Optional[Path]:
    for root in search_roots:
        if not root.exists():
            continue
        direct = root / filename
        if direct.is_file():
            return direct
        for p in root.rglob(filename):
            if p.is_file():
                return p
    return None

def resolve_model_path(hf_repo_id: str, keywords: List[str]) -> str:
    """
    Checks if the model is mounted as a Kaggle Model under /kaggle/input first
    (ensuring PyTorch weights exist and skipping ONNX subfolders);
    otherwise falls back to HuggingFace Hub ID.
    """
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for cfg_path in kaggle_input.rglob("config.json"):
            parent = cfg_path.parent
            if parent.name.lower() == "onnx":
                continue
            has_pt = (parent / "model.safetensors").exists() or (parent / "pytorch_model.bin").exists()
            parent_str = str(parent).lower()
            if has_pt and all(k.lower() in parent_str for k in keywords):
                print(f"[ModelPath] Found mounted Kaggle model at: {parent}")
                return str(parent)
    print(f"[ModelPath] Using Hugging Face Hub ID: {hf_repo_id}")
    return hf_repo_id

SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("../data/raw"),
    Path("../data/processed"),
    Path("data/raw"),
    Path("data/processed"),
    Path("."),
]

TRAIN_PATH = find_file_in_dirs("train.csv", SEARCH_ROOTS)
TEST_PATH = find_file_in_dirs("test.csv", SEARCH_ROOTS)
SAMPLE_SUB_PATH = find_file_in_dirs("sample_submission.csv", SEARCH_ROOTS)
FOLDS_PATH = find_file_in_dirs("folds.csv", SEARCH_ROOTS)

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../data/processed")
SUBMISSION_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../submissions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = resolve_model_path(
    hf_repo_id=CHOSEN_MODEL_NAME,
    keywords=["deberta", "prompt", "injection"],
)

print(f"TRAIN_PATH      : {TRAIN_PATH}")
print(f"TEST_PATH       : {TEST_PATH}")
print(f"FOLDS_PATH      : {FOLDS_PATH}")
print(f"OUTPUT_DIR      : {OUTPUT_DIR}")

In [ ]:
# ============================================================================
# 3. SAFE DATA LOADING & SHARED 5-FOLD ALIGNMENT
# ============================================================================
def clean_text_safety(
    text: str,
    normalize_nfc: bool = True,
    strip_null_bytes: bool = True,
    preserve_case: bool = False,
) -> str:
    """
    Exact copy of src/data_loader.py:clean_text_safety.
    Preserves Romance diacritics (pt, fr, it) and adversarial delimiters.
    """
    if pd.isna(text) or text is None:
        return ""
    if not isinstance(text, str):
        text = str(text)
    if normalize_nfc:
        text = unicodedata.normalize("NFC", text)
    if strip_null_bytes:
        text = text.replace("\x00", "")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    if not preserve_case:
        text = text.lower()
    return text.strip()

def get_shared_folds(train_df: pd.DataFrame, folds_path: Optional[Path]) -> np.ndarray:
    """
    Loads folds.csv if uploaded, otherwise reproduces the exact deterministic
    StratifiedGroupKFold split from src/folds.py (seed=42).
    """
    if folds_path is not None and folds_path.exists():
        f_df = pd.read_csv(folds_path, dtype={"id": str})
        assert len(f_df) == len(train_df), "folds.csv row count mismatch!"
        assert (f_df["id"].values == train_df["id"].astype(str).values).all(), "folds.csv ID order mismatch!"
        print(f"[Folds] Loaded official team folds from {folds_path}")
        return f_df["fold"].values

    print("[Folds] folds.csv not found in input; regenerating deterministic 5-fold split (src/folds.py)...")
    key = train_df["text"].fillna("").astype(str).map(lambda s: re.sub(r"\s+", " ", clean_text_safety(s, preserve_case=False)))
    groups = pd.factorize(key)[0]
    folds = np.full(len(train_df), -1)
    y = train_df["label"].values
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for f, (_, va) in enumerate(splitter.split(train_df, y, groups)):
        folds[va] = f
    return folds

train_df = pd.read_csv(TRAIN_PATH, encoding="utf-8-sig", keep_default_na=False, na_values=[""], dtype={"id": str})
test_df = pd.read_csv(TEST_PATH, encoding="utf-8-sig", keep_default_na=False, na_values=[""], dtype={"id": str})
folds = get_shared_folds(train_df, FOLDS_PATH)
train_df["fold"] = folds
train_df["target"] = train_df["label"].map(LABEL2ID)

# Prepare model input text with language prefix (preserve_case=True for DeBERTa-v3)
train_df["input_text"] = (
    "lang: " + train_df["language"].astype(str) + " | " +
    train_df["text"].map(lambda s: clean_text_safety(s, preserve_case=True))
)
test_df["input_text"] = (
    "lang: " + test_df["language"].astype(str) + " | " +
    test_df["text"].map(lambda s: clean_text_safety(s, preserve_case=True))
)

print(f"Train shape: {train_df.shape} | Test shape: {test_df.shape}")
print("Fold distribution:\n", pd.crosstab(train_df["fold"], train_df["label"]))

In [ ]:
# ============================================================================
# 4. HEAD + TAIL TOKENIZATION & DYNAMIC PADDING DATASET
# ============================================================================
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)

def encode_head_tail(texts: List[str], tokenizer, max_len: int = 512, head_len: int = 256, tail_len: int = 256) -> List[List[int]]:
    """
    Tokenizes texts without truncation first. If token length > max_len,
    keeps the first `head_len` tokens (including CLS & language tag) and
    the last `tail_len` tokens (including end-of-prompt injection payload & SEP).
    """
    all_ids = tokenizer(texts, add_special_tokens=True, truncation=False, verbose=False)["input_ids"]
    processed = []
    truncated_count = 0
    for ids in all_ids:
        if len(ids) > max_len:
            ids = ids[:head_len] + ids[-tail_len:]
            truncated_count += 1
        processed.append(ids)
    print(f"[Tokenizer] Processed {len(texts):,} texts | Head+Tail truncated (> {max_len} tokens): {truncated_count:,}")
    return processed

print("Tokenizing train & test sets with Head+Tail truncation...")
train_input_ids = encode_head_tail(train_df["input_text"].tolist(), tokenizer, MAX_LEN, HEAD_LEN, TAIL_LEN)
test_input_ids = encode_head_tail(test_df["input_text"].tolist(), tokenizer, MAX_LEN, HEAD_LEN, TAIL_LEN)

class PromptAttackDataset(Dataset):
    def __init__(self, input_ids_list: List[List[int]], labels: Optional[np.ndarray] = None):
        self.input_ids_list = input_ids_list
        self.labels = labels

    def __len__(self) -> int:
        return len(self.input_ids_list)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        item = {"input_ids": self.input_ids_list[idx]}
        if self.labels is not None:
            item["labels"] = int(self.labels[idx])
        return item

class DynamicPaddingCollator:
    def __init__(self, pad_token_id: int):
        self.pad_token_id = pad_token_id

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        max_batch_len = max(len(x["input_ids"]) for x in batch)
        input_ids = []
        attention_mask = []
        for x in batch:
            ids = x["input_ids"]
            pad_len = max_batch_len - len(ids)
            input_ids.append(ids + [self.pad_token_id] * pad_len)
            attention_mask.append([1] * len(ids) + [0] * pad_len)

        out = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        }
        if "labels" in batch[0]:
            out["labels"] = torch.tensor([x["labels"] for x in batch], dtype=torch.long)
        return out

collator = DynamicPaddingCollator(pad_token_id=tokenizer.pad_token_id or 0)

In [ ]:
# ============================================================================
# 5. TRAINING & INFERENCE ENGINE (CLASS-WEIGHTED LOSS + AMP FP16)
# ============================================================================
@torch.no_grad()
def predict_proba(model: nn.Module, loader: DataLoader, device: torch.device) -> np.ndarray:
    model.eval()
    all_probs = []
    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs = torch.softmax(logits.float(), dim=-1).cpu().numpy()
        all_probs.append(probs)
    return np.vstack(all_probs)

def train_one_fold(
    fold: int,
    train_ids: List[List[int]],
    y_train: np.ndarray,
    val_ids: List[List[int]],
    y_val: np.ndarray,
    test_loader: DataLoader,
) -> Tuple[np.ndarray, np.ndarray, float]:
    print(f"\n{'='*70}\n🚀 Training Fold {fold} | Model: {CHOSEN_MODEL_NAME}\n{'='*70}")
    seed_everything(SEED + fold)

    train_dataset = PromptAttackDataset(train_ids, y_train)
    val_dataset = PromptAttackDataset(val_ids, y_val)

    train_loader = DataLoader(
        train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True,
        collate_fn=collator, num_workers=0, pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=VALID_BATCH_SIZE, shuffle=False,
        collate_fn=collator, num_workers=0, pin_memory=True
    )

    # ignore_mismatched_sizes=True replaces the 2-class pretrained head with our 3-class head
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH,
        num_labels=3,
        label2id=LABEL2ID,
        id2label=ID2LABEL,
        ignore_mismatched_sizes=True,
    )
    model = model.float().to(DEVICE)

    weight_tensor = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weight_tensor)

    no_decay = ["bias", "LayerNorm.weight", "LayerNorm.bias"]
    optimizer_grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
        },
    ]
    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=LR)
    total_steps = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * WARMUP_RATIO), num_training_steps=total_steps
    )
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_val_f1 = -1.0
    best_val_probs = None
    best_ckpt_path = OUTPUT_DIR / f"best_{OUTPUT_PREFIX}_fold{fold}.pt"

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        model.train()
        running_loss = 0.0
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader, start=1):
            input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
            attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
            labels = batch["labels"].to(DEVICE, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                loss = criterion(logits, labels) / GRAD_ACCUM_STEPS

            scaler.scale(loss).backward()
            running_loss += loss.item() * GRAD_ACCUM_STEPS

            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()

        val_probs = predict_proba(model, val_loader, DEVICE)
        val_preds = val_probs.argmax(axis=1)
        val_macro_f1 = f1_score(y_val, val_preds, average="macro", zero_division=0)
        per_cls_f1 = f1_score(y_val, val_preds, labels=[0, 1, 2], average=None, zero_division=0)
        elapsed = time.time() - t0

        print(
            f"Epoch {epoch}/{EPOCHS} [{elapsed:.1f}s] | "
            f"Train Loss: {running_loss / len(train_loader):.4f} | "
            f"Val Macro-F1: {val_macro_f1:.4f} "
            f"(BENIGN: {per_cls_f1[0]:.4f}, JAILBREAK: {per_cls_f1[1]:.4f}, INJECTION: {per_cls_f1[2]:.4f})"
        )

        if val_macro_f1 > best_val_f1:
            best_val_f1 = val_macro_f1
            best_val_probs = val_probs
            torch.save(model.state_dict(), best_ckpt_path)
            print(f"  --> ⭐ Saved new best checkpoint for Fold {fold} (Macro-F1 = {best_val_f1:.4f})")

    # Load best fold checkpoint to predict on test set
    model.load_state_dict(torch.load(best_ckpt_path, map_location=DEVICE))
    fold_test_probs = predict_proba(model, test_loader, DEVICE)

    # Remove checkpoint file after inference to save Kaggle disk space
    if best_ckpt_path.exists():
        best_ckpt_path.unlink()

    del model, optimizer, scheduler, scaler, train_loader, val_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return best_val_probs, fold_test_probs, best_val_f1

In [ ]:
# ============================================================================
# 6. RUN 5-FOLD CROSS-VALIDATION & EXPORT TEAM HANDSHAKE ARRAYS (.NPY)
# ============================================================================
test_dataset = PromptAttackDataset(test_input_ids, labels=None)
test_loader = DataLoader(
    test_dataset, batch_size=VALID_BATCH_SIZE, shuffle=False,
    collate_fn=collator, num_workers=0, pin_memory=True
)

oof_proba = np.zeros((len(train_df), 3), dtype=np.float64)
test_proba_folds = []
y_all = train_df["target"].values

for fold in FOLD_TO_RUN:
    tr_idx = np.where(folds != fold)[0]
    va_idx = np.where(folds == fold)[0]

    tr_ids = [train_input_ids[i] for i in tr_idx]
    va_ids = [train_input_ids[i] for i in va_idx]

    val_probs, fold_test_probs, _ = train_one_fold(
        fold=fold,
        train_ids=tr_ids,
        y_train=y_all[tr_idx],
        val_ids=va_ids,
        y_val=y_all[va_idx],
        test_loader=test_loader,
    )

    oof_proba[va_idx] = val_probs
    test_proba_folds.append(fold_test_probs)

test_proba = np.mean(test_proba_folds, axis=0)

# Save canonical (14000, 3) OOF and (6000, 3) Test probability matrices for Nhật Tân
oof_save_path = OUTPUT_DIR / f"oof_{OUTPUT_PREFIX}.npy"
test_save_path = OUTPUT_DIR / f"test_{OUTPUT_PREFIX}.npy"
np.save(oof_save_path, oof_proba)
np.save(test_save_path, test_proba)

print(f"\n✅ Saved OOF probabilities  {oof_proba.shape} -> {oof_save_path}")
print(f"✅ Saved Test probabilities {test_proba.shape} -> {test_save_path}")

In [ ]:
# ============================================================================
# 7. OOF MACRO-F1 EVALUATION, MULTIPLIER TUNING & SUBMISSION VALIDATION
# ============================================================================
def optimize_class_multipliers(y_true: np.ndarray, oof_probs: np.ndarray) -> Tuple[np.ndarray, float]:
    """Exact implementation of src/blend.py:optimize_class_multipliers."""
    base_preds = oof_probs.argmax(axis=1)
    base_f1 = float(f1_score(y_true, base_preds, average="macro", zero_division=0))
    best_f1 = base_f1
    best_m = np.ones(3, dtype=float)

    grid_vals = np.linspace(0.3, 3.0, 28)
    for m1 in grid_vals:
        for m2 in grid_vals:
            m_cand = np.array([1.0, m1, m2], dtype=float)
            preds = (oof_probs * m_cand).argmax(axis=1)
            score = float(f1_score(y_true, preds, average="macro", zero_division=0))
            if score > best_f1:
                best_f1 = score
                best_m = m_cand

    def _loss(params: np.ndarray) -> float:
        if np.any(params <= 0.05) or np.any(params > 10.0):
            return 1.0
        m_vec = np.concatenate([[1.0], params])
        preds = (oof_probs * m_vec).argmax(axis=1)
        return -float(f1_score(y_true, preds, average="macro", zero_division=0))

    res = minimize(_loss, x0=best_m[1:], method="Nelder-Mead", options={"maxiter": 300})
    if -float(res.fun) > best_f1:
        best_f1 = -float(res.fun)
        best_m = np.concatenate([[1.0], res.x])

    print(f"[Threshold Tuning] Raw OOF Macro-F1: {base_f1:.4f} -> Tuned OOF Macro-F1: {best_f1:.4f} (+{best_f1 - base_f1:.4f})")
    print(f"[Threshold Tuning] Optimal Class Multipliers: {np.round(best_m, 4).tolist()}")
    return best_m, best_f1

evaluated_mask = np.isin(folds, FOLD_TO_RUN)

# Apply EDA Golden Length Rules (from TASK.md Section 2) before threshold tuning:
# - JAILBREAK length in train is strictly within [112, 3752] chars
# - Prompts > 5128 chars are 100% BENIGN
def apply_length_priors(probs: np.ndarray, texts: pd.Series) -> np.ndarray:
    adj = probs.copy()
    lens = texts.fillna("").astype(str).str.len().values
    adj[lens > 5128, 1] = 0.0
    adj[lens > 5128, 2] = 0.0
    adj[(lens < 112) | (lens > 3752), 1] *= 0.05
    row_sums = adj.sum(axis=1, keepdims=True)
    return np.divide(adj, np.where(row_sums > 0, row_sums, 1.0))

oof_proba_adj = apply_length_priors(oof_proba, train_df["text"])
test_proba_adj = apply_length_priors(test_proba, test_df["text"])

best_multipliers, tuned_oof_f1 = optimize_class_multipliers(
    y_all[evaluated_mask], oof_proba_adj[evaluated_mask]
)

oof_final_preds = (oof_proba_adj[evaluated_mask] * best_multipliers).argmax(axis=1)
print("\n=== OOF Classification Report ===")
print(classification_report(y_all[evaluated_mask], oof_final_preds, target_names=list(VALID_LABELS), digits=4))
print("=== OOF Confusion Matrix ===")
print(confusion_matrix(y_all[evaluated_mask], oof_final_preds))

# Apply optimal multipliers to Test probabilities
test_pred_ids = (test_proba_adj * best_multipliers).argmax(axis=1)
test_pred_labels = [ID2LABEL[i] for i in test_pred_ids]

sub_df = pd.DataFrame({"id": test_df["id"].astype(str), "label": test_pred_labels})

# Strict Submission Validation (Exact rules from src/utils.py:validate_submission)
assert list(sub_df.columns) == ["id", "label"], f"Invalid columns: {list(sub_df.columns)}"
assert len(sub_df) == 6000, f"Expected 6000 rows, got {len(sub_df)}"
assert sub_df.isna().sum().sum() == 0, "Submission contains NaN values!"
assert (sub_df["id"].values == test_df["id"].astype(str).values).all(), "Test ID order mismatch!"
assert set(sub_df["label"].unique()).issubset(set(VALID_LABELS)), f"Invalid label values: {sub_df['label'].unique()}"

sub_path_main = SUBMISSION_DIR / "submission.csv"
sub_path_named = SUBMISSION_DIR / f"submission_{OUTPUT_PREFIX}.csv"
sub_df.to_csv(sub_path_main, index=False)
sub_df.to_csv(sub_path_named, index=False)

print(f"\n🎉 Validated submission saved to: {sub_path_main} and {sub_path_named}")
print("\nPredicted Test Class Distribution:")
print(sub_df["label"].value_counts(normalize=True).mul(100).round(2).astype(str) + "%")